# Phase 2: Data Preprocessing, Cleaning & Transformation
**Project:** Forbes Most Valuable Sports Teams Valuation Analysis  
**Seed:** `random_state = 42`  
**Tasks:**
1. Cleaning: Deduplication, decompose composite strings, strip symbols (`$`, `B`, `M`, `%`).
2. Missing Imputation: Median imputation for member-owned clubs with indicator flags.
3. Outlier Flagging: IQR bounds (flag, don't blindly drop).
4. Feature Selection: Drop raw ID columns, evaluate correlation & variance.
5. Transformation & Discretization: Min-Max, Z-score, and quantile binning (`Low`/`Med`/`High`) to generate `clean.csv`, `scaled.csv`, and `transactions.csv`.


In [1]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

clean_path = os.path.join("..", "data", "clean.csv")
if not os.path.exists(clean_path):
    clean_path = os.path.join("data", "clean.csv")

df_clean = pd.read_csv(clean_path)
print(f"Cleaned dataset loaded. Shape: {df_clean.shape}")
display(df_clean.head())


Cleaned dataset loaded. Shape: (50, 19)


,Owner,Team_Name,Sport,League,Country,Value_USD_B,Five_Year_Change_Pct,Year_Purchased_Clean,Price_Paid_USD_M,Price_Imputed,Year_Imputed,Years_Held,Price_Paid_USD_B,Valuation_Gain_USD_B,Value_USD_B_Outlier,Five_Year_Change_Pct_Outlier,Year_Purchased_Clean_Outlier,Price_Paid_USD_M_Outlier,Years_Held_Outlier
0,Jerry Jones,Dallas Cowboys,American Football,NFL,USA,5.70,43.0,1989.0,150.0,0,0,32.0,0.1500,5.5500,0,0,0,0,0
1,Steinbrenner family,New York Yankees,Baseball,MLB,USA,5.25,54.0,1973.0,8.8,0,0,48.0,0.0088,5.2412,0,0,0,0,0
2,Madison Square Garden Sports,New York Knicks,Basketball,NBA,USA,5.00,67.0,1997.0,300.0,0,0,24.0,0.3000,4.7000,0,0,0,0,0
3,club members,Barcelona,Soccer,La Liga,Spain,4.76,34.0,2000.0,360.0,1,1,21.0,0.3600,4.4000,0,0,0,0,0
4,club members,Real Madrid,Soccer,La Liga,Spain,4.75,30.0,2000.0,360.0,1,1,21.0,0.3600,4.3900,0,0,0,0,0


### 2.1 Derived Financial & Tenure Metrics

In [2]:
display(df_clean[['Value_USD_B', 'Five_Year_Change_Pct', 'Price_Paid_USD_M', 'Years_Held', 'Valuation_Gain_USD_B']].describe())


,Value_USD_B,Five_Year_Change_Pct,Price_Paid_USD_M,Years_Held,Valuation_Gain_USD_B
count,50.000000,50.000000,50.000000,50.000000,50.000000
mean,3.427000,65.160000,605.848164,28.000000,2.821152
std,0.836148,37.773417,742.082002,24.578903,1.276483
min,2.350000,23.000000,0.000100,1.000000,-0.550000
25%,2.812500,43.000000,100.000000,11.250000,2.357500
50%,3.200000,56.000000,360.000000,21.000000,2.900500
75%,4.000000,67.000000,737.500000,36.000000,3.593750
max,5.700000,207.000000,3200.000000,101.000000,5.550000


### 2.2 IQR Outlier Detection & Flagging Summary

In [3]:
outlier_cols = [c for c in df_clean.columns if c.endswith('_Outlier')]
print("Outlier Counts by Feature (Preserved via Boolean Indicators):")
display(df_clean[outlier_cols].sum().to_frame(name='Outlier Count'))


Outlier Counts by Feature (Preserved via Boolean Indicators):


,Outlier Count
Value_USD_B_Outlier,0
Five_Year_Change_Pct_Outlier,6
Year_Purchased_Clean_Outlier,4
Price_Paid_USD_M_Outlier,6
Years_Held_Outlier,4


### 2.3 Feature Variance & Pearson Correlation Matrix

In [4]:
num_cols = ['Value_USD_B', 'Five_Year_Change_Pct', 'Year_Purchased_Clean', 'Price_Paid_USD_M', 'Years_Held']
print("Variance:")
print(df_clean[num_cols].var())

print("\nCorrelation Matrix:")
display(df_clean[num_cols].corr().round(3))


Variance:
Value_USD_B                  0.699144
Five_Year_Change_Pct      1426.831020
Year_Purchased_Clean       604.122449
Price_Paid_USD_M        550685.698203
Years_Held                 604.122449
dtype: float64

Correlation Matrix:


,Value_USD_B,Five_Year_Change_Pct,Year_Purchased_Clean,Price_Paid_USD_M,Years_Held
Value_USD_B,1.000,-0.035,-0.125,-0.306,0.125
Five_Year_Change_Pct,-0.035,1.000,0.160,-0.135,-0.160
Year_Purchased_Clean,-0.125,0.160,1.000,0.568,-1.000
Price_Paid_USD_M,-0.306,-0.135,0.568,1.000,-0.568
Years_Held,0.125,-0.160,-1.000,-0.568,1.000


### 2.4 Scaled Dataset & Transaction Table Verification

In [5]:
trans_path = os.path.join("..", "data", "transactions.csv")
if not os.path.exists(trans_path):
    trans_path = os.path.join("data", "transactions.csv")

df_trans = pd.read_csv(trans_path)
print(f"Transaction Table: {len(df_trans)} transactions")
display(df_trans.head(6))


Transaction Table: 50 transactions


,TID,Items
0,0,"Sport_American Football, League_NFL, Country_U..."
1,1,"Sport_Baseball, League_MLB, Country_USA, Value..."
2,2,"Sport_Basketball, League_NBA, Country_USA, Val..."
3,3,"Sport_Soccer, League_La Liga, Country_Spain, V..."
4,4,"Sport_Soccer, League_La Liga, Country_Spain, V..."
5,5,"Sport_Basketball, League_NBA, Country_USA, Val..."
